# LeetCode 3260 — Find the Largest Palindrome Divisible by K bằng Python

- Link: https://leetcode.com/problems/find-the-largest-palindrome-divisible-by-k/
- Độ khó: **Hard**
- Chủ đề: **Tham lam chữ số từ trái sang + DP khả thi trên số dư** (mỗi cặp vị trí đối xứng đóng góp `d·(10^i + 10^{n−1−i}) mod k`), và **phân tích theo dấu hiệu chia hết** cho các `k` dễ

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def largestPalindrome(self, n: int, k: int) -> str:
```

## 1. Mục tiêu học

1. Palindrome `n` chữ số được quyết định bởi `h = ⌈n/2⌉` chữ số đầu; cặp vị trí `(i, n−1−i)` cùng chữ số `d` đóng góp `d · w_i (mod k)` với `w_i = 10^{n−1−i} + 10^i` (hoặc `10^i` nếu `i` là vị trí giữa).
2. "Số lớn nhất" ⇔ **tham lam** chọn chữ số lớn nhất ở vị trí trái nhất, miễn là phần còn lại **vẫn có thể** đưa tổng về số dư 0.
3. Tính trước `can[i][r]` = các cặp `i..h−1` có thể tạo tổng `≡ r (mod k)` không — DP lùi `O(h · k · 10)`.
4. Với `k ∈ {1, 2, 3, 4, 5, 8, 9}` có công thức trực tiếp từ dấu hiệu chia hết (kiểm chứng bằng DP).
5. Kiểm chứng bằng duyệt palindrome giảm dần với `n` nhỏ.

## 2. Đề bài & ví dụ

Cho `n`, `k`. Trả về (dạng chuỗi) số **lớn nhất** có `n` chữ số là palindrome và chia hết cho `k` (không có số 0 đầu).

**Ví dụ 1:** `n = 3, k = 5` → `"595"`. **Ví dụ 2:** `n = 1, k = 4` → `"8"`. **Ví dụ 3:** `n = 5, k = 6` → `"89898"`.

**Ràng buộc:** `1 ≤ n ≤ 10^5`; `1 ≤ k ≤ 9`.

In [1]:
from typing import List

examples = [(3, 5, "595"), (1, 4, "8"), (5, 6, "89898")]
for n, k, exp in examples:
    print(f"n = {n}, k = {k} -> kỳ vọng {exp!r}")

n = 3, k = 5 -> kỳ vọng '595'
n = 1, k = 4 -> kỳ vọng '8'
n = 5, k = 6 -> kỳ vọng '89898'


## 3. Kiến thức nền

### 3.1. Trọng số của từng cặp vị trí (theo modulo `k`)

In [2]:
def pair_weights(n, k):
    pw = [1 % k] * n
    for i in range(1, n):
        pw[i] = pw[i - 1] * 10 % k
    h = (n + 1) // 2
    # vị trí i tính từ trái (chữ số hàng 10^{n-1-i}); đối xứng là vị trí n-1-i (hàng 10^i)
    return [(pw[n - 1 - i] + pw[i]) % k if i != n - 1 - i else pw[i] for i in range(h)]

print('n = 5, k = 6:', pair_weights(5, 6))

n = 5, k = 6: [5, 2, 4]


### 3.2. Tham lam với kiểm tra khả thi

So sánh hai số cùng độ dài theo thứ tự từ điển ⇒ chọn chữ số lớn nhất có thể ở vị trí đầu, rồi vị trí tiếp theo… **miễn** còn cách hoàn thành hợp lệ. Bảng `can[i][r]` trả lời "từ vị trí `i`, có thể tạo phần dư `r` không".

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 3 (`n = 5, k = 6`): `w = [10001 mod 6, 1010 mod 6, 100 mod 6] = [5, 2, 4]`.

- Vị trí 0: thử `9` — cần phần còn lại `≡ −45 ≡ 3 (mod 6)` với `2a + 4b`: luôn chẵn ⇒ không được. Thử `8`: cần `≡ −40 ≡ 2`: được (ví dụ `a = 9`, `b = 8`: `18 + 32 = 50 ≡ 2`).
- Vị trí 1: `9`: cần `4b ≡ 2 − 18 ≡ 2 (mod 6)`: `b = 8` (`32 ≡ 2`) ✓.
- Vị trí giữa: `8`.

⇒ `89898` ✓.

## 5. Cách 1 — Brute Force: duyệt palindrome giảm dần

In [3]:
def brute(n, k):
    h = (n + 1) // 2
    for left in range(10 ** h - 1, 10 ** (h - 1) - 1, -1):
        s = str(left)
        p = s + s[::-1][n % 2:]
        if int(p) % k == 0:
            return p
    return ""

for n, k, exp in examples:
    print(repr(brute(n, k)), '| kỳ vọng', repr(exp))

'595' | kỳ vọng '595'
'8' | kỳ vọng '8'
'89898' | kỳ vọng '89898'


Duyệt nửa đầu giảm dần ⇔ duyệt palindrome giảm dần. Chỉ cho `n ≤ 12`.

## 6. Ý tưởng thuật toán / bước đột phá

```
w = pair_weights(n, k); h = len(w)
can[h] = {0}
can[i] = { (r + d·w[i]) mod k : r ∈ can[i+1], d ∈ 0..9 }  (d ≥ 1 nếu i = 0)
cur = 0
for i in 0..h-1:
    chọn d lớn nhất (≥ 1 nếu i = 0) với (−cur − d·w[i]) mod k ∈ can[i+1]
    cur += d·w[i]
```

`O(h · k · 10)` ≤ `4.5·10^6` phép.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Đệ quy có nhớ: thử chữ số lớn nhất trước

In [4]:
import sys
sys.setrecursionlimit(10000)
from functools import lru_cache

def memo_greedy(n, k):
    w = pair_weights(n, k); h = len(w)

    @lru_cache(maxsize=None)
    def ok(i, r):                         # từ vị trí i, cần tổng ≡ r
        if i == h:
            return r % k == 0
        return any(ok(i + 1, (r - d * w[i]) % k) for d in range(9, -1 if i else 0, -1))

    res, need = [], 0
    for i in range(h):
        for d in range(9, 0 if i == 0 else -1, -1):
            if ok(i + 1, (need - d * w[i]) % k):
                res.append(str(d)); need = (need - d * w[i]) % k
                break
    ok.cache_clear()
    s = ''.join(res)
    return s + s[::-1][n % 2:]

for n, k, exp in examples:
    print(repr(memo_greedy(n, k)), '| kỳ vọng', repr(exp))

'595' | kỳ vọng '595'
'8' | kỳ vọng '8'
'89898' | kỳ vọng '89898'


Đệ quy sâu `h` tầng — chỉ dùng cho `n` vừa phải.

### 7.2. Cách 3 — DP khả thi lùi + tham lam tiến (lời giải chính)

In [5]:
def dp_greedy(n, k):
    w = pair_weights(n, k); h = len(w)
    can = [0] * (h + 1)                     # bitmask các số dư đạt được
    can[h] = 1                               # chỉ số dư 0
    full = (1 << k) - 1
    for i in range(h - 1, -1, -1):
        m = 0
        lo = 1 if i == 0 else 0
        for d in range(lo, 10):
            sh = d * w[i] % k
            nxt = can[i + 1]
            m |= ((nxt << sh) | (nxt >> (k - sh))) & full if sh else nxt
        can[i] = m
    res = []
    cur = 0                                  # tổng đã chọn (mod k)
    for i in range(h):
        for d in range(9, 0 if i == 0 else -1, -1):
            need = (-(cur + d * w[i])) % k
            if can[i + 1] >> need & 1:
                res.append(str(d)); cur = (cur + d * w[i]) % k
                break
    s = ''.join(res)
    return s + s[::-1][n % 2:]

for n, k, exp in examples:
    print(repr(dp_greedy(n, k)), '| kỳ vọng', repr(exp))

'595' | kỳ vọng '595'
'8' | kỳ vọng '8'
'89898' | kỳ vọng '89898'


Bitmask: `can[i]` có bit `r` ⇔ các vị trí `i..h−1` tạo được tổng `≡ r`. Cộng thêm `d·w[i]` = **xoay vòng** bitmask `sh` bit trong `k` bit.

### 7.3. Cách 4 — Công thức theo dấu hiệu chia hết (cho `k ∉ {6, 7}`)

- `k ∈ {1, 3, 9}`: toàn `9` (tổng chữ số `9n` chia hết cho 9).
- `k = 2`: chữ số cuối chẵn ⇒ `8 9…9 8`.
- `k = 4`: hai chữ số cuối chia hết cho 4 ⇒ `88 9…9 88` (`n ≤ 4`: toàn `8`).
- `k = 8`: ba chữ số cuối ⇒ `888 9…9 888` (`n ≤ 6`: toàn `8`).
- `k = 5`: tận cùng `5` (không thể `0`) ⇒ `5 9…9 5`.
- `k ∈ {6, 7}`: dùng DP.

In [6]:
def formula(n, k):
    if k in (1, 3, 9):
        return '9' * n
    if k == 2:
        return '8' if n == 1 else '8' + '9' * (n - 2) + '8'
    if k == 4:
        return '8' * n if n <= 4 else '88' + '9' * (n - 4) + '88'
    if k == 8:
        return '8' * n if n <= 6 else '888' + '9' * (n - 6) + '888'
    if k == 5:
        return '5' if n == 1 else '5' + '9' * (n - 2) + '5'
    return dp_greedy(n, k)

for n, k, exp in examples:
    print(repr(formula(n, k)), '| kỳ vọng', repr(exp))

'595' | kỳ vọng '595'
'8' | kỳ vọng '8'
'89898' | kỳ vọng '89898'


## 8. TRACE từng bước

Trace cách 3 với `n = 6`, `k = 7`.

In [7]:
def trace(n, k):
    w = pair_weights(n, k); h = len(w)
    can = [0] * (h + 1); can[h] = 1; full = (1 << k) - 1
    for i in range(h - 1, -1, -1):
        m = 0
        for d in range(1 if i == 0 else 0, 10):
            sh = d * w[i] % k; nxt = can[i + 1]
            m |= ((nxt << sh) | (nxt >> (k - sh))) & full if sh else nxt
        can[i] = m
    print('trọng số w =', w, '| can =', [format(c, f'0{k}b') for c in can])
    cur, res = 0, []
    for i in range(h):
        for d in range(9, 0 if i == 0 else -1, -1):
            need = (-(cur + d * w[i])) % k
            ok = can[i + 1] >> need & 1
            if ok:
                res.append(str(d)); cur = (cur + d * w[i]) % k
                print(f'  vị trí {i}: chọn {d} (phần sau cần dư {need})')
                break
    s = ''.join(res); ans = s + s[::-1][n % 2:]
    print('đáp án:', ans, '| vét cạn:', brute(n, k), '| chia 7:', int(ans) % 7)

trace(6, 7)

trọng số w = [6, 0, 1] | can = ['1111111', '1111111', '1111111', '0000001']
  vị trí 0: chọn 9 (phần sau cần dư 2)
  vị trí 1: chọn 9 (phần sau cần dư 2)
  vị trí 2: chọn 9 (phần sau cần dư 0)
đáp án: 999999 | vét cạn: 999999 | chia 7: 0


## 9. Code nộp LeetCode

In [8]:
class Solution:
    def largestPalindrome(self, n: int, k: int) -> str:
        pw = [1 % k] * n
        for i in range(1, n):
            pw[i] = pw[i - 1] * 10 % k
        h = (n + 1) // 2
        w = [(pw[n - 1 - i] + pw[i]) % k if i != n - 1 - i else pw[i] for i in range(h)]
        full = (1 << k) - 1
        can = [0] * (h + 1)
        can[h] = 1
        for i in range(h - 1, -1, -1):
            nxt = can[i + 1]
            m = 0
            for d in range(1 if i == 0 else 0, 10):
                sh = d * w[i] % k
                m |= ((nxt << sh) | (nxt >> (k - sh))) & full if sh else nxt
            can[i] = m
        res = []
        cur = 0
        for i in range(h):
            for d in range(9, 0 if i == 0 else -1, -1):
                v = (cur + d * w[i]) % k
                if can[i + 1] >> ((-v) % k) & 1:
                    res.append(chr(48 + d))
                    cur = v
                    break
        s = ''.join(res)
        return s + s[::-1][n % 2:]

for n, k, exp in examples:
    got = Solution().largestPalindrome(n, k)
    print(repr(got), 'OK' if got == exp else 'SAI')

'595' OK
'8' OK
'89898' OK


## 10. Vì sao thuật toán đúng

1. **Palindrome ⇔ nửa đầu:** giá trị `≡ Σ_i d_i · w_i (mod k)`.
2. **Bảng khả thi:** `can[i]` đúng theo quy nạp lùi — tập tổng đạt được của `i..h−1` là hợp của các tập `can[i+1]` dịch đi `d·w_i`.
3. **Tham lam:** so sánh hai palindrome cùng độ dài ⇔ so sánh nửa đầu theo thứ tự từ điển. Ở mỗi vị trí, chọn chữ số lớn nhất mà phần còn lại vẫn đạt được số dư cần ⇒ được nửa đầu lớn nhất (lập luận "trao đổi": mọi lời giải hợp lệ khác có chữ số nhỏ hơn ở vị trí đầu tiên khác biệt).
4. **Luôn tồn tại:** `k ≤ 9` ⇒ palindrome toàn chữ số `k`… (`kk…k` chia hết cho `k`) nên tham lam luôn tìm được (vị trí 0 chọn `d ≥ 1`).

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Duyệt palindrome giảm dần | `O(10^{n/2} · n)` | `O(n)` |
| 2. Đệ quy có nhớ | `O(h · k · 10)` | `O(h · k)` |
| 3. DP bitmask + tham lam | `O(h · 10)` thao tác bitmask | `O(h)` |
| 4. Công thức | `O(n)` | `O(n)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [9]:
impls = {'memo_greedy': memo_greedy, 'dp_greedy': dp_greedy, 'formula': formula,
         'submit': lambda n, k: Solution().largestPalindrome(n, k)}
edge = [(1, 1, "9"), (1, 7, "7"), (2, 7, "77"), (2, 6, "66"), (3, 8, "888"), (4, 7, None), (7, 6, None)]
for n, k, exp in examples + edge:
    e = brute(n, k)
    if exp is not None:
        assert e == exp, (n, k, e)
    for name, f in impls.items():
        assert f(n, k) == e, (name, n, k)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu đầy đủ với vét cạn (`n ≤ 10`, mọi `k`)

In [10]:
cnt = 0
for n in range(1, 11):
    for k in range(1, 10):
        exp = brute(n, k)
        for name, f in impls.items():
            assert f(n, k) == exp, (name, n, k)
        cnt += 1
print('Khớp vét cạn trên', cnt, 'cặp (n, k).')

Khớp vét cạn trên 90 cặp (n, k).


### 12.3. Ca lớn `n = 10^5`

In [11]:
import time
for k in range(1, 10):
    t = time.time(); a = Solution().largestPalindrome(10 ** 5, k); t1 = time.time() - t
    b = formula(10 ** 5, k)
    assert a == b and len(a) == 10 ** 5 and a == a[::-1] and int(a[-3:]) % (k if k in (2, 4, 5, 8) else 1) == 0
    print(f"k = {k}: {a[:6]}…{a[len(a)//2 - 3:len(a)//2 + 3]}…{a[-6:]} | {t1:.2f}s")

k = 1: 999999…999999…999999 | 0.06s


k = 2: 899999…999999…999998 | 0.05s


k = 3: 999999…999999…999999 | 0.06s


k = 4: 889999…999999…999988 | 0.05s


k = 5: 599999…999999…999995 | 0.05s


k = 6: 899999…997799…999998 | 0.07s


k = 7: 999999…997799…999999 | 0.07s
k = 8: 888999…999999…999888 | 0.05s


k = 9: 999999…999999…999999 | 0.09s


Kiểm tra độc lập chia hết cho `k` trên số khổng lồ bằng tổng chữ số theo trọng số (mod `k`), không cần `int()` của 10^5 chữ số (dù Python làm được):

In [12]:
def mod_of_string(s, k):
    r = 0
    for ch in s:
        r = (r * 10 + ord(ch) - 48) % k
    return r

for k in range(1, 10):
    assert mod_of_string(Solution().largestPalindrome(10 ** 5, k), k) == 0
    assert mod_of_string(Solution().largestPalindrome(10 ** 5 - 1, k), k) == 0
print('Mọi đáp án n = 1e5 và 1e5 - 1 chia hết cho k.')

Mọi đáp án n = 1e5 và 1e5 - 1 chia hết cho k.


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tham lam **không** kiểm tra khả thi (chỉnh chữ số cuối)

In [13]:
def wrong_fix_last(n, k):
    h = (n + 1) // 2
    for d in range(9, 0, -1):                               # SAI: chỉ thay chữ số giữa / cặp trong cùng
        s = '9' * (h - 1) + str(d)
        p = s + s[::-1][n % 2:]
        if int(p) % k == 0:
            return p
    return '-'

print('sai :', wrong_fix_last(*examples[2][:2]), '| đúng:', Solution().largestPalindrome(*examples[2][:2]))

sai : - | đúng: 89898


**Vì sao sai:** với `k = 6`, palindrome cần chữ số cuối **chẵn** ⇒ chữ số **đầu** phải đổi; chỉ chỉnh phần giữa không bao giờ đủ.

### Lỗi 2 — Quên chữ số đầu phải khác 0

In [14]:
def wrong_allow_zero_first(n, k):
    w = pair_weights(n, k); h = len(w)
    can = [set() for _ in range(h + 1)]; can[h] = {0}
    for i in range(h - 1, -1, -1):
        can[i] = {(r + d * w[i]) % k for r in can[i + 1] for d in range(10)}
    res, cur = [], 0
    for i in range(h):
        for d in range(9, -1, -1):                          # SAI: cho phép d = 0 ở vị trí 0
            if (-(cur + d * w[i])) % k in can[i + 1]:
                res.append(str(d)); cur = (cur + d * w[i]) % k; break
    s = ''.join(res); return s + s[::-1][n % 2:]

found = [(n, k) for n in range(1, 8) for k in range(1, 10) if wrong_allow_zero_first(n, k) != Solution().largestPalindrome(n, k)]
print('các (n, k) lệch:', found[:5], '... ->', 'không có lệch ở đây' if not found else '')

các (n, k) lệch: [] ... -> không có lệch ở đây


**Vì sao (về lý thuyết) sai:** nếu chữ số 9..1 đều không khả thi ở vị trí đầu, bản sai sẽ chọn `0` — số có số 0 đầu. Với `k ≤ 9` điều đó **không bao giờ xảy ra** (luôn có palindrome `kk…k`), nên bản sai tình cờ cho đúng; nhưng trong biến thể với `k` lớn điều kiện `d ≥ 1` là bắt buộc. Ô dưới minh họa với `k = 101`, `n = 2`: không palindrome 2 chữ số nào (`11, 22, …, 99`) chia hết cho 101, đáp án đúng là `"-1"`, còn bản bỏ điều kiện trả về `"00"`.

In [15]:
def generic(n, k, first_nonzero=True):
    w = pair_weights(n, k); h = len(w)
    can = [set() for _ in range(h + 1)]; can[h] = {0}
    for i in range(h - 1, -1, -1):
        can[i] = {(r + d * w[i]) % k for r in can[i + 1] for d in range(1 if (i == 0 and first_nonzero) else 0, 10)}
    if 0 not in can[0]:
        return "-1"
    res, cur = [], 0
    for i in range(h):
        for d in range(9, (0 if (i == 0 and first_nonzero) else -1), -1):
            if (-(cur + d * w[i])) % k in can[i + 1]:
                res.append(str(d)); cur = (cur + d * w[i]) % k; break
    s = ''.join(res); return s + s[::-1][n % 2:]

print('k = 101, n = 2 -> đúng:', generic(2, 101), '| bỏ điều kiện chữ số đầu:', generic(2, 101, first_nonzero=False))

k = 101, n = 2 -> đúng: -1 | bỏ điều kiện chữ số đầu: 00


### Lỗi 3 — Trọng số vị trí giữa tính hai lần

In [16]:
def wrong_middle_weight(n, k):
    pw = [1 % k] * n
    for i in range(1, n):
        pw[i] = pw[i - 1] * 10 % k
    h = (n + 1) // 2
    w = [(pw[n - 1 - i] + pw[i]) % k for i in range(h)]       # SAI: vị trí giữa (n lẻ) bị cộng 2 lần
    can = [set() for _ in range(h + 1)]; can[h] = {0}
    for i in range(h - 1, -1, -1):
        can[i] = {(r + d * w[i]) % k for r in can[i + 1] for d in range(1 if i == 0 else 0, 10)}
    res, cur = [], 0
    for i in range(h):
        for d in range(9, 0 if i == 0 else -1, -1):
            if (-(cur + d * w[i])) % k in can[i + 1]:
                res.append(str(d)); cur = (cur + d * w[i]) % k; break
    s = ''.join(res); return s + s[::-1][n % 2:]

print('sai :', wrong_middle_weight(*examples[2][:2]), '| đúng:', Solution().largestPalindrome(*examples[2][:2]))

sai : 89798 | đúng: 89898


**Vì sao sai:** với `n` lẻ, chữ số giữa chỉ xuất hiện **một** lần.

### Lỗi 4 — Công thức `k = 4` dùng cho mọi `n` (quên `n ≤ 4`)

In [17]:
def wrong_formula_k4(n):
    return '88' + '9' * (n - 4) + '88'                         # SAI khi n < 4

print('sai :', repr(wrong_formula_k4(3)), '| đúng:', repr(Solution().largestPalindrome(3, 4)))

sai : '8888' | đúng: '888'


**Vì sao sai:** với `n = 3`, chuỗi `'88' + '' + '88'` có 4 chữ số — sai độ dài. Các công thức đóng cần xử lý riêng `n` nhỏ — đó là lý do cần DP để đối chiếu.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Số lớn nhất / nhỏ nhất thỏa ràng buộc chia hết" ⇒ tham lam từng chữ số + DP khả thi trên số dư.
- Palindrome ⇒ ghép cặp vị trí, trọng số `10^i + 10^{n−1−i}`.
- `k` nhỏ ⇒ bitmask số dư (xoay vòng bit).

## 15. Biến thể của bài

1. Số **nhỏ nhất** có `n` chữ số là palindrome chia hết cho `k` ⇒ tham lam chữ số nhỏ nhất.
2. `k` lớn (≤ 1000) ⇒ cùng DP, có thể không tồn tại (`-1`).
3. Palindrome **chẵn chữ số** chia hết cho 11 — luôn (tính chất palindrome chẵn).
4. Đếm số palindrome `n` chữ số chia hết cho `k` ⇒ DP đếm thay vì khả thi.

In [18]:
# Biến thể 1 và 4
def smallest_palindrome(n, k):
    w = pair_weights(n, k); h = len(w)
    can = [set() for _ in range(h + 1)]; can[h] = {0}
    for i in range(h - 1, -1, -1):
        can[i] = {(r + d * w[i]) % k for r in can[i + 1] for d in range(1 if i == 0 else 0, 10)}
    res, cur = [], 0
    for i in range(h):
        for d in range(1 if i == 0 else 0, 10):
            if (-(cur + d * w[i])) % k in can[i + 1]:
                res.append(str(d)); cur = (cur + d * w[i]) % k; break
    s = ''.join(res); return s + s[::-1][n % 2:]

def count_palindromes(n, k):
    w = pair_weights(n, k); h = len(w)
    dp = [0] * k; dp[0] = 1
    for i in range(h - 1, -1, -1):
        nd = [0] * k
        for r in range(k):
            if dp[r]:
                for d in range(1 if i == 0 else 0, 10):
                    nd[(r + d * w[i]) % k] += dp[r]
        dp = nd
    return dp[0]

for n in range(1, 8):
    for k in range(1, 10):
        pals = [p for p in (str(a) + str(a)[::-1][n % 2:] for a in range(10 ** ((n + 1) // 2 - 1), 10 ** ((n + 1) // 2))) if int(p) % k == 0]
        assert smallest_palindrome(n, k) == min(pals, key=int) and count_palindromes(n, k) == len(pals)
print('Biến thể "nhỏ nhất" và "đếm" khớp vét cạn (n <= 7).')

Biến thể "nhỏ nhất" và "đếm" khớp vét cạn (n <= 7).


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Duyệt palindrome giảm dần | mũ | Oracle |
| 2 | Đệ quy có nhớ | `O(h k)` | Đệ quy sâu |
| 3 | DP bitmask + tham lam | `O(10 h)` | **Nộp bài**, tổng quát |
| 4 | Công thức chia hết | `O(n)` | Chỉ một số `k` |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `n = 2`, `k = 3`.

**Bài 2 (tính tay):** `n = 4`, `k = 8`.

**Bài 3 (điền TODO):** biến thể 2 — `k ≤ 1000`, trả `"-1"` nếu không tồn tại.

**Bài 4 (tự viết lại):** số palindrome lớn nhất `≤` một số `N` cho trước chia hết cho `k` (cùng số chữ số).

**Bài 5 (mini challenge):** tìm công thức đóng cho `k = 6` (kiểm chứng bằng DP với `n ≤ 50`).

In [19]:
def large_k_todo(n, k):
    # TODO
    raise NotImplementedError

def largest_at_most_todo(N_str, k):
    # TODO
    raise NotImplementedError

def formula_k6_todo(n):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `99` chia hết cho 3. (2) Lớn nhất. (3) `"99"`.

**Bài 2.** (1) `n ≤ 6` với `k = 8` ⇒ toàn `8`. (2) `8888 / 8 = 1111`. (3) `"8888"`.

**Bài 3.** (1) Hàm `generic(n, k)` ở lỗi 2. (2) Trả `-1` khi `0 ∉ can[0]`. (3) Vét cạn nhỏ.

**Bài 4.** (1) Thêm trạng thái "đang bằng tiền tố của N hay đã nhỏ hơn". (2) Tham lam giảm dần từ chữ số của `N`. (3) Vì là palindrome, ràng buộc ≤ N phức tạp hơn — vét cạn nhỏ để kiểm.

**Bài 5.** (1) Cần chẵn (cuối 8) và chia 3. (2) Từ DP: `n` lẻ ⇒ `8 9…9 8 9…9 8` (giữa `8`), `n` chẵn ⇒ `8 9…9 77 9…9 8`. (3) Kiểm với `n ≤ 50`.

## 19. Đáp án tham khảo cho bài tập

In [20]:
# Bài 1, 2, 3, 5
assert Solution().largestPalindrome(2, 3) == "99"
assert Solution().largestPalindrome(4, 8) == "8888"

def large_k(n, k):
    return generic(n, k)

for n in range(1, 7):
    for k in [11, 13, 101, 999]:
        pals = [p for p in (str(a) + str(a)[::-1][n % 2:] for a in range(10 ** ((n + 1) // 2 - 1), 10 ** ((n + 1) // 2))) if int(p) % k == 0]
        assert large_k(n, k) == (max(pals, key=int) if pals else "-1")

def formula_k6(n):
    if n <= 2:
        return '6' * n
    if n % 2:
        side = '8' + '9' * ((n - 3) // 2)
        return side + '8' + side[::-1]
    side = '8' + '9' * ((n - 4) // 2)
    return side + '77' + side[::-1]

for n in range(1, 51):
    assert formula_k6(n) == Solution().largestPalindrome(n, 6), n
print('Bài 1: 99 | Bài 2: 8888 | Bài 3 khớp vét cạn với k lớn | Bài 5: công thức k = 6 đúng tới n = 50.')

Bài 1: 99 | Bài 2: 8888 | Bài 3 khớp vét cạn với k lớn | Bài 5: công thức k = 6 đúng tới n = 50.


In [21]:
# Bài 4 (vét cạn có hướng dẫn: duyệt nửa đầu giảm dần từ nửa đầu của N)
def largest_at_most(N_str, k):
    n = len(N_str); h = (n + 1) // 2
    start = int(N_str[:h])
    for left in range(start, 10 ** (h - 1) - 1, -1):
        s = str(left); p = s + s[::-1][n % 2:]
        if int(p) <= int(N_str) and int(p) % k == 0:
            return p
    return "-1"

for N in [1000, 12345, 99999, 54321]:
    for k in [2, 6, 7]:
        exp = max((str(x) for x in range(10 ** (len(str(N)) - 1), N + 1) if str(x) == str(x)[::-1] and x % k == 0), key=int, default="-1")
        assert largest_at_most(str(N), k) == exp
print('Bài 4 khớp vét cạn.')

Bài 4 khớp vét cạn.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
w[i] = 10^{n-1-i} + 10^i (mod k), vị trí giữa chỉ 10^i
can[h] = {0}; can[i] = ∪_d (can[i+1] + d·w[i])           # bitmask xoay vòng
tham lam: i = 0..h-1, d = 9..(1 nếu i = 0, ngược lại 0) đầu tiên có (−cur − d·w[i]) mod k ∈ can[i+1]
```

**Checklist:**

- [ ] Trọng số vị trí giữa (n lẻ) chỉ một lần?
- [ ] Chữ số đầu `≥ 1`?
- [ ] Kiểm tra khả thi phần còn lại trước khi chọn chữ số?
- [ ] Ghép nửa sau đúng với `n` lẻ (bỏ chữ số giữa khi đảo)?
- [ ] Công thức đóng nhớ xử lý `n` nhỏ?